In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from gspread_pandas import Spread, conf
import time
import sys
import re

In [1]:
CONFIG_PATH = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
logistic_sheet_url = "https://docs.google.com/spreadsheets/d/1eo2tY_57lcOTtGOyU5GOz2IQON3b0VsVf8rTsbfT3HM/edit"


In [2]:
def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z2000']) 
        spreadsheet.df_to_sheet(df, index=False, replace=False, headers=False, start="A2")

In [ ]:
# Cell 1: Imports
import pandas as pd
import re
from datetime import datetime
from gspread_pandas import Spread, conf 

# Cell 2: Configuration
CONFIG_PATH = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
LOGISTIC_SHEET_URL_FEB = "https://docs.google.com/spreadsheets/d/1S6WaI1rXuf5ogN44ph8weA_z_bMCg85U3uaZ7RHvpfM/edit?gid=1671266520#gid=1671266520"
LOGISTIC_SHEET_URL_MAR = "https://docs.google.com/spreadsheets/d/1S6WaI1rXuf5ogN44ph8weA_z_bMCg85U3uaZ7RHvpfM/edit?gid=1671266520#gid=1671266520"
FINANCIAL_SHEET_URL = "https://docs.google.com/spreadsheets/d/1wTjaEYO8VCVu5eZxEXAb25Ed_7E320EWpi3PSqDTEAQ/edit?gid=0#gid=0"

# Manual Vendor Mapping
MANUAL_VENDOR_MAP = {
    'OUD LOVERS': 'LPG',
    'INTENSE SIGNATURE': 'LPG',
    'ARBE PURO COMBO': 'LPG',
    'CHERIE BLOSSOM': 'LPG',
    'VELORA POP HEART': 'LPG',
    'VELORA SUGAR BLISS': 'LPG',
    'VELORA VIVA CHOCO': 'LPG',
    'ASTORIA': 'LPG',
    'JENAN': 'LPG',
    'NAJAH PISTACHIO': 'LPG',
    'LEON': 'LPG',
    'OPUS': 'LPG',
    'ENIGMA': 'LPG',
    'RANIA': 'LPG',
    'PREMIUM EDITION': 'OUD AL SALAM',
    'ABSOLUTE MOUNTAIN AVENUE': 'OUD AL SALAM',
    'SEVEN DAY': 'RT',
    'OLD MEMORIES': 'RT',
    'ARCHER COMBO': 'ATYAF'
}

# Cell 3: Helper Functions
def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z10000']) 
        upload_df = df.astype(str)

        spreadsheet.df_to_sheet(upload_df, index=False, replace=False, headers=False, start="A2")

def clean_and_sum_digits(value):
    val_str = str(value).strip().lower()
    if not val_str or val_str == 'nan':
        return 0
    numbers = re.findall(r'\d+', val_str)
    total = 0
    for num in numbers:
        if len(num) > 3 and len(num) % 3 == 0:
            chunks = [int(num[i:i+3]) for i in range(0, len(num), 3)]
            total += sum(chunks)
        else:
            total += int(num)
    return total


now_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(f"[{now_ts}] Starting sync process...")
        
# Authentication
c = conf.get_config(file_name=CONFIG_PATH)
spread = Spread(LOGISTIC_SHEET_URL_MAR, config=c)
# fin_spread = Spread(FINANCIAL_SHEET_URL, config=c)

REQUIRED_COLUMNS = [
    'Country', 'AGENT', 'DATE', 'TRACKING NUMBER', 'EM NUMBER',
    'NAME', 'NUMBER1', 'NUMBER2', 'STATE / CITY', 'ADDRESS',
    'CUSTOMER PATH', 'STATUS', 'DISPATCHED DATE',
    'REASON', 'DELIVERY AGENT'
]

# Map actual sheet column names → clean names
COLUMN_RENAME_MAP = {
    'TRACKING \nNUMBER': 'TRACKING NUMBER',
    'EMNUMBER':          'EM NUMBER',
    'CUSTOMER\nPATH':    'CUSTOMER PATH',
    'DISPATCHED\nDATE':  'DISPATCHED DATE',
    'NATIONAL \nCODE':   'NATIONAL CODE',
}

def load_and_standardize(spread, sheet_name, country_value):
    df = spread.sheet_to_df(sheet=sheet_name, index=None)
    df.columns = df.columns.astype(str).str.strip()
    df = df.loc[:, ~df.columns.duplicated()]

    # Rename messy column names to clean names
    df = df.rename(columns=COLUMN_RENAME_MAP)

    df['Country'] = country_value

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ''

    df = df[REQUIRED_COLUMNS]

    # Keep only rows where AGENT and EM NUMBER have actual data
    agent_has_data = df['AGENT'].astype(str).str.strip().replace('nan', '').ne('')
    em_has_data = df['EM NUMBER'].astype(str).str.strip().replace('nan', '').ne('')
    df = df[agent_has_data & em_has_data]

    return df.reset_index(drop=True)

# 1. Load the DataFrames with standardized columns
KSA_DF   = load_and_standardize(spread, "ORDER LIST - KSA",     "KSA")
UAE_DF   = load_and_standardize(spread, "ORDER LIST - UAE",     "UAE")
QATAR_DF = load_and_standardize(spread, "ORDER LIST - QATAR",   "Qatar")
BH_DF    = load_and_standardize(spread, "ORDER LIST - BAHRAIN", "Bahrain")

# 2. Merge (concatenate)
df = pd.concat([KSA_DF, UAE_DF, QATAR_DF, BH_DF], ignore_index=True)
df.to_excel("./data.xlsx")

[2026-03-05 12:07:16] Starting sync process...


In [30]:
df

,Country,AGENT,DATE,TRACKING NUMBER,EM NUMBER,NAME,NUMBER1,NUMBER2,STATE / CITY,ADDRESS,CUSTOMER PATH,STATUS,DISPATCHED DATE,REASON,DELIVERY AGENT
0,KSA,AMINA,02/03/2026,SA823851077708,EMCR36019,Anzar,502476663,,Dammam,"956X+FQ8, 2nd Industrial City, Dammam 34226, S...",LEAD,DISPATCHED,03/03/2026,,
1,KSA,AMINA,02/03/2026,SA823851077709,EMCR36032,Sameer💐🌹,552379674,,Riyadh,"RJ5P+HXG As Sahafah, Riyadh Saudi Arabia",LEAD,DISPATCHED,03/03/2026,,
2,KSA,AMINA,02/03/2026,,EMCR36039,Lazy Shams,536404568,,Riyadh,"JGHQ+6W2, Tabuk, Dhahrat Laban, Riyadh 13784, ...",LEAD,,,,
3,KSA,AMINA,02/03/2026,,EMCR36046,Jakariya Munsi,576433698,,Al Safa,"H6G4+3XR Al-Safa, Jeddah Saudi Arabia",LEAD,,,,
4,KSA,AMINA,02/03/2026,SA823851077710,EMCR36049,Ashiq Avunhikatt,537423833,,Jeddah,"F5PR+F9M, Al-Balad, Jeddah 22236, Saudi Arabia",LEAD,DISPATCHED,03/03/2026,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,Bahrain,RINSIYA,02/03/2026,S38360,EMCR50029,Hassainar M,36024510,"36024510, 39602423",Jannusan,"6GG2+6JQ Jannusan, Bahrain, https://maps.app.g...",,DISPATCH,,,
113,Bahrain,SAFAN,02/03/2026,,EMCR34063,Ananthu Rajeevan,917356425564,,East Riffa,"Building 100, Road 3901, Riffa Al Hajjiyat, Bl...",,ON HOLD,,Please update Local number,
114,Bahrain,SAFAN,02/03/2026,S38361,EMCR34064,kunju soofi,36083687,,Manama Center,"Road 213 ,Manama, Bahrain",,DISPATCH,,,
115,Bahrain,ZAKIYA,02/03/2026,S38362,EMCR32032,Pratheesh Kumar,38873980,,Maameer,Shaikh Jaber Al Ahmed Al Subah Highway\r\nAl M...,,DISPATCH,,,


In [3]:
# Cell 1: Imports
import pandas as pd
import re
from datetime import datetime
from gspread_pandas import Spread, conf 

# Cell 2: Configuration
CONFIG_PATH = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
LOGISTIC_SHEET_URL = "https://docs.google.com/spreadsheets/d/1S6WaI1rXuf5ogN44ph8weA_z_bMCg85U3uaZ7RHvpfM/edit?gid=1671266520#gid=1671266520"
FINANCIAL_SHEET_URL = "https://docs.google.com/spreadsheets/d/1wTjaEYO8VCVu5eZxEXAb25Ed_7E320EWpi3PSqDTEAQ/edit?gid=0#gid=0"

# Manual Vendor Mapping
MANUAL_VENDOR_MAP = {
    'OUD LOVERS': 'LPG',
    'INTENSE SIGNATURE': 'LPG',
    'ARBE PURO COMBO': 'LPG',
    'CHERIE BLOSSOM': 'LPG',
    'VELORA POP HEART': 'LPG',
    'VELORA SUGAR BLISS': 'LPG',
    'VELORA VIVA CHOCO': 'LPG',
    'ASTORIA': 'LPG',
    'JENAN': 'LPG',
    'NAJAH PISTACHIO': 'LPG',
    'LEON': 'LPG',
    'OPUS': 'LPG',
    'ENIGMA': 'LPG',
    'RANIA': 'LPG',
    'PREMIUM EDITION': 'OUD AL SALAM',
    'ABSOLUTE MOUNTAIN AVENUE': 'OUD AL SALAM',
    'SEVEN DAY': 'RT',
    'OLD MEMORIES': 'RT',
    'ARCHER COMBO': 'ATYAF'
}

# Cell 3: Helper Functions
def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z10000']) 
        upload_df = df.astype(str)

        spreadsheet.df_to_sheet(upload_df, index=False, replace=False, headers=False, start="A2")

def clean_and_sum_digits(value):
    val_str = str(value).strip().lower()
    if not val_str or val_str == 'nan':
        return 0
    numbers = re.findall(r'\d+', val_str)
    total = 0
    for num in numbers:
        if len(num) > 3 and len(num) % 3 == 0:
            chunks = [int(num[i:i+3]) for i in range(0, len(num), 3)]
            total += sum(chunks)
        else:
            total += int(num)
    return total

# Cell 4: Main Processing Logic
def process_and_sync():
    try:
        now_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{now_ts}] Starting sync process...")
        
        # Authentication
        c = conf.get_config(file_name=CONFIG_PATH)
        spread = Spread(LOGISTIC_SHEET_URL, config=c)
        fin_spread = Spread(FINANCIAL_SHEET_URL, config=c)
        
        # 1. Load the DataFrames
        KSA_DF = spread.sheet_to_df(sheet="ORDER LIST - KSA", index=None)
        UAE_DF = spread.sheet_to_df(sheet="ORDER LIST - UAE", index=None)
        QATAR_DF = spread.sheet_to_df(sheet="ORDER LIST - QATRA", index=None)
        BH_DF = spread.sheet_to_df(sheet="ORDER LIST - BAHRAIN", index=None)

        # 2. Add the 'Country' column to each
        KSA_DF['Country'] = 'KSA'
        UAE_DF['Country'] = 'UAE'
        QATAR_DF['Country'] = 'Qatar'
        BH_DF['Country'] = 'Bahrain'

        # 3. Merge (concatenate)
        df = pd.concat([KSA_DF, UAE_DF, QATAR_DF, BH_DF], ignore_index=True)
        print(display(df))
        # Load Data
        df = spread.sheet_to_df(sheet="Order list", index=None)
        
        if df.empty:
            print(f"[{now_ts}] Warning: Data Frame is empty. Skipping.")
            return None, None, None

        # Basic Cleaning
        df.columns = df.columns.astype(str).str.strip().str.upper()

        if 'DATE' not in df.columns:
            print(f"Error: 'DATE' column missing.")
            return None, None, None
            
        df['DATE'] = pd.to_datetime(df['DATE'], dayfirst=True, errors='coerce').dt.date
        df = df.dropna(subset=['DATE'])

        if 'TRACKING NUMBER' in df.columns:
            df = df[df['TRACKING NUMBER'].astype(str).str.strip() != ""]

        cat_cols = ['STATUS', 'DELIVERY AGENT', 'COUNTRY', 'PRODUCT1']
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip().str.upper()

        # Apply Manual Vendor Mapping
        df['VENDOR'] = df['PRODUCT1'].map(MANUAL_VENDOR_MAP).fillna('UNKNOWN VENDOR')

        # --- 1. LOGISTICS SUMMARY REPORT ---
        report = pd.DataFrame({'DATE': sorted(df['DATE'].unique())})
        report.set_index('DATE', inplace=True)
        date_groups = df.groupby('DATE')
        report['TOTAL WON ORDERS'] = date_groups.size()

        for country in ['KSA', 'UAE', 'QATAR', 'BAHRAIN']:
            report[f'{country} ORDERS'] = df[df['COUNTRY'] == country].groupby('DATE').size()

        agents = ['TAWSEEL', 'NAQEL', 'RGS', 'FETCH SAUDI', 'FETCH QATAR', 'FETCH BAHRAIN', 'JNT']
        for agent in agents:
            report[agent] = df[df['DELIVERY AGENT'] == agent].groupby('DATE').size()

        # Specific Agent-Country Combos
        combos = {
            'KSA-RGS': ('RGS', 'KSA'),
            'KSA-NAQEL': ('NAQEL', 'KSA'),
            'KSA-FETCH_SAUDI': ('FETCH SAUDI', 'KSA'),
            'UAE-TAWSEEL': ('TAWSEEL', 'UAE'),
            'QATAR-FETCH_QATAR': ('FETCH QATAR', 'QATAR'),
            'BAHRAIN-FETCH_BAHRAIN': ('FETCH BAHRAIN', 'BAHRAIN')
        }
        for col_name, (agent, country) in combos.items():
            report[col_name] = df[(df['DELIVERY AGENT'] == agent) & (df['COUNTRY'] == country)].groupby('DATE').size()


        statuses = ['DELIVERED AND UNPAID', 'OUT FOR DELIVERY', 'RTO', 'ON HOLD', 'NO ANSWER', 'CANCELLED', 'ORDER CONFIRMED']
        for status in statuses:
            report[status] = df[df['STATUS'] == status].groupby('DATE').size()
        
        # Dispatched logic
        dispatched_mask = ~df['STATUS'].isin(['DELIVERED AND UNPAID', 'CANCELLED', 'RTO'])
        report['DISPATCHED'] = df[dispatched_mask].groupby('DATE').size()

        report = report.fillna(0).astype(int).reset_index()

        # --- 2. PRODUCT & FINANCIAL REPORTS ---
        if 'TOTAL' in df.columns:
            df['TOTAL'] = df['TOTAL'].apply(clean_and_sum_digits)
            merge_keys = ['COUNTRY', 'DELIVERY AGENT', 'VENDOR', 'PRODUCT1']
            
            # Base Stats
            report_detailed = df.groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            report_detailed.columns = merge_keys + ['TOTAL_ORDERS', 'TOTAL_REVENUE']

            # Delivered
            delivered_mask = df['STATUS'] == 'DELIVERED AND UNPAID'
            delivered_detailed = df[delivered_mask].groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            delivered_detailed.columns = merge_keys + ['DELIVERED_TOTAL_ORDERS', 'DELIVERED_REVENUE']

            # RTO
            rto_mask = df['STATUS'] == 'RTO'
            rto_detailed = df[rto_mask].groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            rto_detailed.columns = merge_keys + ['RTO_TOTAL_ORDERS', 'RTO_REVENUE']

            # Cancelled
            cancelled_mask = df['STATUS'] == 'CANCELLED'
            cancelled_detailed = df[cancelled_mask].groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            cancelled_detailed.columns = merge_keys + ['CANCELLED_TOTAL_ORDERS', 'CANCELLED_REVENUE']

            # Merge Logic
            merged_df = report_detailed.merge(delivered_detailed, on=merge_keys, how='left')
            merged_df = merged_df.merge(rto_detailed, on=merge_keys, how='left')
            merged_df = merged_df.merge(cancelled_detailed, on=merge_keys, how='left')

            fill_cols = [
                'DELIVERED_TOTAL_ORDERS', 'DELIVERED_REVENUE', 
                'RTO_TOTAL_ORDERS', 'RTO_REVENUE', 
                'CANCELLED_TOTAL_ORDERS', 'CANCELLED_REVENUE'
            ]
            merged_df[fill_cols] = merged_df[fill_cols].fillna(0).astype(int)
            
            # Summary by Date & Vendor
            report_by_country = df.groupby(['DATE', 'COUNTRY', 'VENDOR', 'DELIVERY AGENT', 'PRODUCT1']).size().reset_index(name='COUNTRY_SALES')

        else:
            merged_df = pd.DataFrame() 
            report_by_country = pd.DataFrame()

        # --- UPLOAD SECTION ---
        # Uncomment these lines to actually push data to Google Sheets
        # safe_upload(spread, report, "Logistic_Summary_Report")
        # safe_upload(fin_spread, report, "Logistic_Summary_Report")
        # safe_upload(fin_spread, merged_df, "Product_Consolidated_Report")
        # safe_upload(fin_spread, report_by_country, "Product_Summary_Report")

        print(f"[{datetime.now().strftime('%H:%M:%S')}] Sync Successful.")
        return report, merged_df, report_by_country

    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Error: {str(e)}")
        return None, None, None

# Cell 5: Run Execution and Display Results
log_report, detailed_report, country_report = process_and_sync()

# View the results in the notebook
if log_report is not None:
    display(log_report.head())

[2026-03-04 12:09:42] Starting sync process...
[12:10:47] Error: Reindexing only valid with uniquely valued Index objects


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from gspread_pandas import Spread, conf
import time
import sys
import re

# --- CONFIGURATION ---
CONFIG_PATH = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
LOGISTIC_SHEET_URL = "https://docs.google.com/spreadsheets/d/1eo2tY_57lcOTtGOyU5GOz2IQON3b0VsVf8rTsbfT3HM/edit"
FINANCIAL_SHEET_URL = "https://docs.google.com/spreadsheets/d/1wTjaEYO8VCVu5eZxEXAb25Ed_7E320EWpi3PSqDTEAQ/edit?gid=0#gid=0"
INTERVAL_MINUTES = 1

# Load Vendor Mapping from Excel
VENDOR_FILE_PATH = r'C:\Users\anudi\Downloads\PersonalProjects\Emarathglobal\Product - Price list.xlsx'

def get_vendor_mapping():
    try:
        # Loading columns A and B (Vendor and Product Name)
        v_df = pd.read_excel(VENDOR_FILE_PATH)
        # Standardize keys and values
        v_df['PRODUCT NAME'] = v_df['PRODUCT NAME'].astype(str).str.strip().str.upper()
        v_df['VENDOR'] = v_df['VENDOR'].astype(str).str.strip().str.upper()
        return dict(zip(v_df['PRODUCT NAME'], v_df['VENDOR']))
    except Exception as e:
        print(f"Error loading Vendor Excel: {e}")
        return {}

def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        # Clear existing data to avoid overlap
        spreadsheet.sheet.batch_clear(['A2:Z10000']) 
        upload_df = df.astype(str)
        # Uploading data starting from A2
        spreadsheet.df_to_sheet(upload_df, index=False, replace=False, headers=False, start="A2")

def clean_and_sum_digits(value):
    val_str = str(value).strip().lower()
    if not val_str or val_str == 'nan':
        return 0
    
    numbers = re.findall(r'\d+', val_str)
    total = 0
    for num in numbers:
        if len(num) > 3 and len(num) % 3 == 0:
            chunks = [int(num[i:i+3]) for i in range(0, len(num), 3)]
            total += sum(chunks)
        else:
            total += int(num)
    return total

def process_and_sync():
    try:
        now_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{now_ts}] Starting sync process...")
        
        # Load fresh mapping each cycle in case Excel changed
        vendor_map = get_vendor_mapping()
        
        c = conf.get_config(file_name=CONFIG_PATH)
        spread = Spread(LOGISTIC_SHEET_URL, config=c)
        fin_spread = Spread(FINANCIAL_SHEET_URL, config=c)
        
        df = spread.sheet_to_df(sheet="Order list", index=None)
        
        if df.empty:
            print(f"[{now_ts}] Warning: Data Frame is empty. Skipping.")
            return

        df.columns = df.columns.astype(str).str.strip().str.upper()

        if 'DATE' not in df.columns:
            print(f"Error: 'DATE' column missing.")
            return
            
        df['DATE'] = pd.to_datetime(df['DATE'], dayfirst=True, errors='coerce').dt.date
        df = df.dropna(subset=['DATE'])

        if 'TRACKING NUMBER' in df.columns:
            df = df[df['TRACKING NUMBER'].astype(str).str.strip() != ""]

        # Standardize categorical columns
        cat_cols = ['STATUS', 'DELIVERY AGENT', 'COUNTRY', 'PRODUCT1']
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip().str.upper()

        # --- VENDOR MAPPING ---
        df['VENDOR'] = df['PRODUCT1'].map(vendor_map).fillna('UNKNOWN VENDOR')

        # --- LOGISTICS SUMMARY REPORT ---
        report = pd.DataFrame({'DATE': sorted(df['DATE'].unique())})
        report.set_index('DATE', inplace=True)

        date_groups = df.groupby('DATE')
        report['TOTAL WON ORDERS'] = date_groups.size()

        # Fill Countries, Agents, and Statuses as before...
        for country in ['KSA', 'UAE', 'QATAR', 'BAHRAIN']:
            report[f'{country} ORDERS'] = df[df['COUNTRY'] == country].groupby('DATE').size()

        agents = ['TAWSEEL', 'NAQEL', 'RGS', 'FETCH SAUDI', 'FETCH QATAR', 'FETCH BAHRAIN', 'JNT']
        for agent in agents:
            report[agent] = df[df['DELIVERY AGENT'] == agent].groupby('DATE').size()

        statuses = ['DELIVERED AND UNPAID', 'OUT FOR DELIVERY', 'RTO', 'ON HOLD', 'NO ANSWER', 'CANCELLED', 'ORDER CONFIRMED']
        for status in statuses:
            report[status] = df[df['STATUS'] == status].groupby('DATE').size()
        
        report = report.fillna(0).astype(int).reset_index()

        # --- FINANCIAL / PRODUCT CONSOLIDATED REPORTS ---
        if 'TOTAL' in df.columns:
            df['TOTAL'] = df['TOTAL'].apply(clean_and_sum_digits)
            
            # Grouping keys now include VENDOR
            merge_keys = ['COUNTRY', 'DELIVERY AGENT', 'VENDOR', 'PRODUCT1']
            
            # 1. Base Report
            report_detailed = df.groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            report_detailed.columns = merge_keys + ['TOTAL_ORDERS', 'TOTAL_REVENUE']

            # 2. Delivered Data
            delivered_mask = df['STATUS'] == 'DELIVERED AND UNPAID'
            delivered_detailed = df[delivered_mask].groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            delivered_detailed.columns = merge_keys + ['DELIVERED_TOTAL_ORDERS', 'DELIVERED_REVENUE']

            # 3. RTO Data
            rto_mask = df['STATUS'] == 'RTO'
            rto_detailed = df[rto_mask].groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            rto_detailed.columns = merge_keys + ['RTO_TOTAL_ORDERS', 'RTO_REVENUE']

            # 4. Cancelled Data
            cancelled_mask = df['STATUS'] == 'CANCELLED'
            cancelled_detailed = df[cancelled_mask].groupby(merge_keys)['TOTAL'].agg(['size', 'sum']).reset_index()
            cancelled_detailed.columns = merge_keys + ['CANCELLED_TOTAL_ORDERS', 'CANCELLED_REVENUE']

            # 5. Final Merge
            merged_df = report_detailed.merge(delivered_detailed, on=merge_keys, how='left')
            merged_df = merged_df.merge(rto_detailed, on=merge_keys, how='left')
            merged_df = merged_df.merge(cancelled_detailed, on=merge_keys, how='left')

            fill_cols = [
                'DELIVERED_TOTAL_ORDERS', 'DELIVERED_REVENUE', 
                'RTO_TOTAL_ORDERS', 'RTO_REVENUE', 
                'CANCELLED_TOTAL_ORDERS', 'CANCELLED_REVENUE'
            ]
            merged_df[fill_cols] = merged_df[fill_cols].fillna(0).astype(int)
            
            # Summary by Vendor/Country/Agent
            report_by_country = df.groupby(['DATE', 'COUNTRY', 'VENDOR', 'DELIVERY AGENT', 'PRODUCT1']).size().reset_index(name='COUNTRY_SALES')

        else:
            merged_df = pd.DataFrame() 
            report_by_country = pd.DataFrame()

        # --- UPLOAD ---
        # safe_upload(spread, report, "Logistic_Summary_Report")
        # safe_upload(fin_spread, report, "Logistic_Summary_Report")
        # safe_upload(fin_spread, merged_df, "Product_Consolidated_Report")
        # safe_upload(fin_spread, report_by_country, "Product_Summary_Report")

        print(f"[{datetime.now().strftime('%H:%M:%S')}] Sync Successful.")

    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Error: {str(e)}")

if __name__ == "__main__":
    print("Scheduler started. Interval: 1 Minute.")
    try:
        while True:
            process_and_sync()
            time.sleep(INTERVAL_MINUTES * 60)
    except KeyboardInterrupt:
        print("\nStopping...")
        sys.exit(0)